# Finansiële data ontleding met Python

**NGRDA150 - Leereenheid 2.2: Datainsamelingsmetodes**

---

## Leereenheid 2.2: Eksterne databronne

**Doel:** Om te leer hoe om data vanaf API's te trek en groot hoeveelhede inligting outomaties te verwerk.


## Inleiding

Rekeningkundiges moet ons dikwels groot hoeveelhede finansiële data verwerk. Hierdie notaboek demonstreer hoe Python gebruik kan word om:

- Markdata outomaties vanaf die internet te trek
- Veelvuldige bates gelyktydig te waardeer
- Historiese tendense te analiseer
- Akkurate, herhaalbare verslae te genereer

Dit is soortgelyk aan hoe ons vir Boland Meubels se data-analise sal werk, maar dan met finansiële markinligting.

---
# Biblioteek gebruik
Ons gaan drie Python-biblioteke gebruik:

yfinance: Om markdata te kry

---

## Stap 1: Kry inligting vir 'n enkele aandeel

Ons moet eers 'n "konneksie" na die data maak deur die simbool (ticker) te gebruik.

Elke maatskappy op die aandelemark het 'n unieke simbool, byvoorbeeld:
- **AAPL** = Apple Inc.
- **MSFT** = Microsoft Corporation
- **GOOGL** = Alphabet Inc. (Google)

In [ ]:
import yfinance as yf

# Ons kies 'n maatskappy (bv. Apple)
simbool = 'AAPL'
aandeel = yf.Ticker(simbool)
#print(f"aandeel = {aandeel}")

# Trek die basiese inligting
inligting = aandeel.info

# Gee vir die studente 'n telling van hoeveel datapunte beskikbaar is
datapunte_telling = len(inligting)
print(f"Sukses! Ons het {datapunte_telling} verskillende inligting-punte vir {simbool} gekry.")
print(f"\nMaatskappy: {inligting.get('longName')}")
print(f"Huidige Markprys: ${inligting.get('currentPrice'):.2f}")
print(f"Markkapitalisasie: ${inligting.get('marketCap'):,}")
print(f"Sektor: {inligting.get('sector')}")

### 🔍 Ondersoek die data struktuur

Die `info` objek is 'n **woordeboek** (dictionary) wat sleutel-waarde pare bevat. Kom ons kyk na 'n paar interessante datapunte:

In [ ]:
# Vertoon die eerste 10 sleutels in die woordeboek
print("Eerste 10 beskikbare data-velde:")
print("="*50)
for i, sleutel in enumerate(list(inligting.keys())[:10]):
    print(f"{i+1}. {sleutel}")

---

## Stap 2: Verwerk 'n portefeulje (Die "Loop")

In rekeningkunde is ons selde geïnteresseerd in net een transaksie. Ons wil 'n lys van bates waardeer. Hier gebruik ons 'n `for`-loop om deur 'n lys te gaan.

**Dink aan dit soos 'n grootboek:** Elke reël verteenwoordig 'n ander bate wat gewaardeer moet word.

In [ ]:
# 'n Lys van aandele in ons kliënt se portefeulje
portefeulje = ['AAPL', 'MSFT', 'TSLA', 'GOOGL', 'AMZN']

print("PORTEFEULJE WAARDASIE VERSLAG")
print("="*70)
print(f"{'Simbool':<10} {'Naam':<30} {'Prys':<15}")
print("-"*70)

# Ons loop deur elke item in die lys
for ticker in portefeulje:
    data = yf.Ticker(ticker)

    # Trek spesifieke data uit die woordeboek
    naam = data.info.get('shortName', 'Onbekend')
    prys = data.info.get('currentPrice', 0.00)

    # Druk die reël in ons "grootboek"
    print(f"{ticker:<10} {naam:<30} ${prys:>12.2f}")

print("-"*70)
print(f"Verslag gegenereer vir {len(portefeulje)} bates.")

### 💡 Uitbreiding: Bereken totale portefeulje waarde

Kom ons voeg 'n berekening by om die totale waarde van die portefeulje te bereken as ons weet hoeveel aandele ons van elk besit:

In [ ]:
# Portefeulje met aantal aandele
portefeulje_detail = {
    'AAPL': 50,
    'MSFT': 30,
    'TSLA': 20,
    'GOOGL': 25,
    'AMZN': 15
}

print("GEDETAILLEERDE PORTEFEULJE WAARDASIE")
print("="*90)
print(f"{'Simbool':<10} {'Naam':<30} {'Aantal':<10} {'Prys':<15} {'Totaal':<15}")
print("-"*90)

totale_waarde = 0

for ticker, aantal in portefeulje_detail.items():
    data = yf.Ticker(ticker)
    naam = data.info.get('shortName', 'Onbekend')
    prys = data.info.get('currentPrice', 0.00)
    totaal = aantal * prys
    totale_waarde += totaal

    print(f"{ticker:<10} {naam:<30} {aantal:<10} ${prys:>12.2f} ${totaal:>12.2f}")

print("="*90)
print(f"{'TOTALE PORTEFEULJE WAARDE:':<60} ${totale_waarde:>12.2f}")
print("="*90)

---

## Stap 3: Analiseer historiese data

Historiese tendense is nuttig vir ouditering of finansiële beplanning. Die API maak dit baie maklik om data vir verskillende tydperke te kry.

Ons kan verskeie tydperke gebruik:
- `1d` = Een dag
- `5d` = Vyf dae
- `1mo` = Een maand
- `3mo` = Drie maande
- `1y` = Een jaar
- `5y` = Vyf jaar

In [ ]:
# Kry die laaste 5 dae se data
simbool = 'AAPL'
aandeel = yf.Ticker(simbool)
historiese_data = aandeel.history(period="5d")

print(f"Historiese pryse vir {simbool} (Laaste 5 dae):")
print("="*60)
print(historiese_data[['Sluit', 'Volume']])

### 📊 Visualiseer die historiese data

Grafieke help ons om tendense vinnig te identifiseer. Kom ons skep 'n eenvoudige lyngrafiek:

In [ ]:
import matplotlib.pyplot as plt

# Kry 3 maande se data
historiese_data_3m = aandeel.history(period="3mo")

# Skep die grafiek
plt.figure(figsize=(12, 6))
plt.plot(historiese_data_3m.index, historiese_data_3m['Close'], linewidth=2, color='blue')
plt.title(f'{simbool} - Sluitingsprys oor Laaste 3 Maande', fontsize=16, fontweight='bold')
plt.xlabel('Datum', fontsize=12)
plt.ylabel('Prys (USD)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Bereken basiese statistieke
print("\nBASIESE STATISTIEKE:")
print("="*40)
print(f"Gemiddelde Prys: ${historiese_data_3m['Close'].mean():.2f}")
print(f"Hoogste Prys: ${historiese_data_3m['Close'].max():.2f}")
print(f"Laagste Prys: ${historiese_data_3m['Close'].min():.2f}")
print(f"Standaard Afwyking: ${historiese_data_3m['Close'].std():.2f}")

---

## Stap 5: Omskakel na Suid-Afrikaanse Rand (ZAR)

As Suid-Afrikaanse rekeningkundiges moet ons dikwels buitelandse geldeenhede na Rand omskakel vir finansiële state.

Ons sal 'n funksie skep wat die wisselkoers outomaties kry en die omskakeling doen.

In [ ]:
# Kry die huidige USD/ZAR wisselkoers
wisselkoers_ticker = yf.Ticker('ZAR=X')
wisselkoers_data = wisselkoers_ticker.history(period='1d')
usd_na_zar = wisselkoers_data['Close'].iloc[-1]

print(f"Huidige Wisselkoers: $1 USD = R{usd_na_zar:.2f} ZAR")
print("="*70)

In [ ]:
# Funksie om USD na ZAR om te skakel
def skakel_na_zar(usd_bedrag, wisselkoers):
    """
    Skakel 'n bedrag in USD om na ZAR.

    Parameters:
    usd_bedrag (float): Bedrag in Amerikaanse Dollars
    wisselkoers (float): Huidige USD/ZAR wisselkoers

    Returns:
    float: Bedrag in Suid-Afrikaanse Rand
    """
    return usd_bedrag * wisselkoers

# Toets die funksie
toets_bedrag = 1000
zar_bedrag = skakel_na_zar(toets_bedrag, usd_na_zar)
print(f"\nVoorbeeld: ${toets_bedrag:.2f} USD = R{zar_bedrag:.2f} ZAR")

### Portefeulje waardasie in ZAR

Kom ons pas ons portefeulje-verslag aan om waardes in beide USD en ZAR te wys:

In [ ]:
print("PORTEFEULJE WAARDASIE (USD & ZAR)")
print("="*110)
print(f"{'Simbool':<10} {'Naam':<25} {'Aantal':<10} {'Prys (USD)':<15} {'Totaal (USD)':<15} {'Totaal (ZAR)':<20}")
print("-"*110)

totale_waarde_usd = 0
totale_waarde_zar = 0

for ticker, aantal in portefeulje_detail.items():
    data = yf.Ticker(ticker)
    naam = data.info.get('shortName', 'Onbekend')[:25]  # Beperk naam lengte
    prys = data.info.get('currentPrice', 0.00)
    totaal_usd = aantal * prys
    totaal_zar = skakel_na_zar(totaal_usd, usd_na_zar)

    totale_waarde_usd += totaal_usd
    totale_waarde_zar += totaal_zar

    print(f"{ticker:<10} {naam:<25} {aantal:<10} ${prys:>12.2f} ${totaal_usd:>12.2f} R{totaal_zar:>15.2f}")

print("="*110)
print(f"{'TOTALE PORTEFEULJE WAARDE:':<55} ${totale_waarde_usd:>12.2f} R{totale_waarde_zar:>15.2f}")
print("="*110)
print(f"\nWisselkoers gebruik: $1 USD = R{usd_na_zar:.4f} ZAR")